In [11]:
from pathlib import Path
import numpy as np
import torch

from data_utils import seed_everything, build_datasets
from evaluation import evaluate_model
from Models.AF_mamba import AFMamba
from Models.baselines import TCN_LSTMModel, TCN_OnlyModel, TCN_TransformerModel

DATA_PATH = Path("Data/structured_dataset_1hz.pt")
FOLD_PATH = Path("Data/subject_folds.pt")
CHECKPOINT_ROOT = Path("Trained_Models")
BATCH_SIZE = 16
PREDICTION_HORIZON = 3600
SEED = 42

MODEL_NAME = "af_mamba"  # Select the model for ablation evaluation.

INPUT_LENGTHS = {
    "5min": 300,
    "10min": 600,
    "30min": 1800,
    "60min": 3600,
}

MODEL_REGISTRY = {
    "af_mamba": lambda: AFMamba(),
    "tcn_bilstm": lambda: TCN_LSTMModel(),
    "tcn_transformer": lambda: TCN_TransformerModel(),
    "tcn_baseline": lambda: TCN_OnlyModel(),
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(SEED)

data = torch.load(DATA_PATH, weights_only=False)
fold_data = torch.load(FOLD_PATH, weights_only=False)
af_sets, nsr_sets = fold_data["af_sets"], fold_data["nsr_sets"]

ablation_results = {}

model_fn = MODEL_REGISTRY[MODEL_NAME]
ablation_results = {}

for input_name, input_size in INPUT_LENGTHS.items():
    print(f"\n========== {MODEL_NAME.upper()} | {input_name.upper()} ==========")
    fold_results = []

    for fold_idx in range(5):
        print(f"\n===== FOLD {fold_idx} =====")

        _, val_loader, test_loader, *_ = build_datasets(
            data,
            af_sets,
            nsr_sets,
            fold_idx,
            input_segment_size=input_size,
            prediction_horizon=PREDICTION_HORIZON,
            batch_size=BATCH_SIZE
        )

        model = model_fn().to(device)

        checkpoint = (
            CHECKPOINT_ROOT
            / MODEL_NAME
            / input_name
            / f"fold_{fold_idx}.pt"
        )

        model.load_state_dict(
            torch.load(
                checkpoint,
                map_location=device,
                weights_only=True
            )
        )

        val_metrics, *_ = evaluate_model(
            model,
            val_loader,
            device=device,
            threshold=None
        )

        threshold = val_metrics["threshold"]

        test_metrics, *_ = evaluate_model(
            model,
            test_loader,
            device=device,
            threshold=threshold
        )

        print(
            f"Sens={test_metrics['recall']:.4f}, "
            f"Spec={test_metrics['specificity']:.4f}, "
            f"F1={test_metrics['f1']:.4f}, "
            f"AUROC={test_metrics['roc_auc']:.4f}, "
            f"AUPRC={test_metrics['auprc']:.4f}"
        )

        fold_results.append(test_metrics)

    ablation_results[input_name] = fold_results

METRICS = {
    "Sensitivity": "recall",
    "Specificity": "specificity",
    "F1": "f1",
    "AUROC": "roc_auc",
    "AUPRC": "auprc",
}

print(f"\n========== {MODEL_NAME.upper()} ABLATION SUMMARY ==========")

for input_name, results in ablation_results.items():
    print(f"\n{input_name}")
    for label, key in METRICS.items():
        values = np.array([r[key] for r in results], dtype=float)
        print(f"{label}: {values.mean():.4f} ± {values.std(ddof=0):.4f}")


========== AF_MAMBA | 5MIN ==========

===== FOLD 0 =====

=== FOLD 0 ===
Subjects: train=140, val=46, test=46
Sens=0.9600, Spec=0.3925, F1=0.3707, AUROC=0.8866, AUPRC=0.7272

===== FOLD 1 =====

=== FOLD 1 ===
Subjects: train=140, val=46, test=46
Sens=0.8444, Spec=0.7977, F1=0.5588, AUROC=0.8726, AUPRC=0.5303

===== FOLD 2 =====

=== FOLD 2 ===
Subjects: train=139, val=47, test=46
Sens=0.9841, Spec=0.8080, F1=0.7381, AUROC=0.9597, AUPRC=0.8945

===== FOLD 3 =====

=== FOLD 3 ===
Subjects: train=138, val=47, test=47
Sens=0.9074, Spec=0.7509, F1=0.5600, AUROC=0.9106, AUPRC=0.7692

===== FOLD 4 =====

=== FOLD 4 ===
Subjects: train=139, val=46, test=47
Sens=0.8636, Spec=0.6259, F1=0.4021, AUROC=0.8319, AUPRC=0.6046

========== AF_MAMBA | 10MIN ==========

===== FOLD 0 =====

=== FOLD 0 ===
Subjects: train=140, val=46, test=46
Sens=0.9600, Spec=0.6717, F1=0.5189, AUROC=0.9394, AUPRC=0.8485

===== FOLD 1 =====

=== FOLD 1 ===
Subjects: train=140, val=46, test=46
Sens=0.7111, Spec=0.8511, 